# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`

This notebook guides you through exploring and processing the 
**A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa**
using the `mlcroissant` library. We use only entity `@id` references for all dataset components (record sets, fields, columns, etc.) as provided by the Croissant schema.

### Dataset Source
The dataset is defined and described via a Croissant schema URL. All code uses `mlcroissant` for programmatic access.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We begin by loading the dataset metadata using `mlcroissant`. This provides programmatic access to the dataset's structure, including all record sets and field descriptors by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL:
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View the dataset metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}")


## 2. Data Overview

Let's review the available record sets (tables) and the fields (columns) within each, along with their `@id` values.

We will print out the record set `@id`s and their field `@id`s for reference. Use these `@id` values for all record and column references going forward.

In [ ]:
# Get all record set `@id`s in the dataset
record_sets = [rs['@id'] for rs in meta.to_json().get('recordSet', [])]
print('Found record sets:')
for rs in meta.to_json().get('recordSet', []):
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '[no name]')}, description: {rs.get('description', '')}")
    print("    Fields/columns:")
    for field in rs.get('field', []):
        print(f"     - @{field.get('@id', '[no id]')}: {field.get('name', '[no name]')}")

# Pick the primary record set for demonstration (assuming there's only one, or choose the main one)
if record_sets:
    main_record_set_id = record_sets[0]
else:
    main_record_set_id = None
print(f"\nMain record set for analysis: {main_record_set_id}")

## 3. Data Extraction

We'll extract records from *all* available record sets, loading them into pandas DataFrames. All access uses their unique `@id` values as keys.

- Use the printed `@id` from the previous overview.

In [ ]:
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set '{rs_id}' with {len(df)} records and fields: {list(df.columns)}\n")

# For demonstration, show the columns & top rows from the main record set
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Fields in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field (by `@id`) from the available columns of the main record set DataFrame for filtering, normalization, and grouping.

The following example demonstrates:
- Filtering records where the numeric field (e.g., 'GAD-7 total score') exceeds a threshold
- Normalizing that field (z-score)
- Grouping data, e.g., by gender, if such a column exists

**Update these `@id` variables as needed for the dataset.**

In [ ]:
# List some column ids to choose a numeric and a group field (update as needed)
df = dataframes[main_record_set_id]
print('Available columns:', df.columns.tolist())

# Use the actual @id from printed columns; guess common ones:
# For demonstration, let's suppose '@gad7_total' and '@gender' are true @id's

# Please change these values to the actual @id names from your data!
numeric_field_id = None
group_field_id = None
# Try to choose the first float/integer field, and the first categorical
for col in df.columns:
    # Heuristics for demonstration; you should check printed column names
    if not numeric_field_id and ('gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower()):
        numeric_field_id = col
    if not group_field_id and ('gender' in col.lower() or 'sex' in col.lower()):
        group_field_id = col

print(f"Using numeric field '@id': {numeric_field_id}")
print(f"Using group field '@id': {group_field_id}")

# Proceed only if the field exists
if numeric_field_id:
    # Drop missing values for the numeric field
    filtered_df = df[df[numeric_field_id].notna()].copy()

    # Choose a threshold (e.g., 10)
    threshold = 10
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized field '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by a field (e.g., gender)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print('No suitable numeric field found for EDA. Please adjust the field selection.')


## 5. Visualization

Let's visualize the distribution of the chosen numeric field in the dataset, and see how it varies by the group field (if present).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print('Numeric field not available for visualization.')


## 6. Conclusion

In this notebook, we:
- Loaded a Croissant-described dataset using `mlcroissant`
- Explored its structure and fields using `@id` for all references
- Loaded each record set into a pandas DataFrame
- Performed basic data analysis and visualization on key numeric fields

This workflow can be extended for deeper statistical or machine learning analysis, always referencing fields by their stable `@id`s for reproducible research. For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).